# B1.4 · Strategic planning and agent allocation

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B1.3 · Threat modelling from the architecture map](https://spbreed.github.io/cyber-commons/lessons/B1.3.html)**.

| | |
|---|---|
| Open-source tooling | Semgrep OSS, CodeQL |
| Open-weight models | GLM-4.6, Llama 3.3 |
| Frontier models | Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

You have four hundred thousand lines and a budget that covers perhaps thirty thousand. Where the effort goes is the single decision with the most leverage in the whole pipeline, and it is usually made by whoever opened the repository first.

## 2 · The framework

```
   finite effort, unequal value

   +----------------------+----------+----------+
   | area                 | reachable| changed  |
   +----------------------+----------+----------+
   | request handling     |   yes    |   yes    |  <- spend here
   | internal utils       |   yes    |   no     |
   | vendored third party |   no     |   no     |  <- and not here
   +----------------------+----------+----------+

   allocation is the highest-leverage decision in the pipeline
```

**Stage 6 — Strategic planning.** You now have a ranked threat model. This stage
decides *where to spend the analysis budget* and *which tool or agent to point at
each target*.

The default is a uniform sweep: run every rule over every file. It is simple,
and it scales cost with repository size rather than with risk — so on a large
monorepo the deep, expensive analysis gets turned off for everything, including
the parts that needed it.

Allocation makes the trade explicit. Three inputs:

- **threat rank** from stage 5,
- **historical risk** from stage 1,
- **tool fit** — rules are cheap and precise on known patterns; model review is
  expensive and finds what rules cannot express (B1.5 measures both).

The output is an assignment: which analyser runs against which boundary, with
what budget. And the honest measure of a good allocation is not coverage — it is
**threat-weighted coverage**, because covering the health endpoint thoroughly is
not an achievement.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 6 — the budget, and two ways to spend it

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Target:
    name: str; file: str; threat_score: int; historical_risk: float; loc: int

TARGETS = [
 Target("load_report → execute", "src/data/reports.py", 12, 0.82, 40),
 Target("store → open",          "src/data/docs.py",     9, 0.91, 30),
 Target("render",                "src/util/render.py",   2, 0.05, 15),
 Target("health",                "src/web/health.py",    1, 0.00, 10),
 Target("upload_doc",            "src/web/handlers.py",  9, 0.30, 60),
 Target("get_report",            "src/web/handlers.py", 11, 0.30, 60),
]

ANALYSERS = {
 # name          cost/100 LOC   finds                         precision
 "grep rules":   (1,   {"CWE-798"},                          0.50),
 "taint rules":  (4,   {"CWE-89", "CWE-78", "CWE-22"},       1.00),
 "model review": (40,  {"CWE-89","CWE-78","CWE-22","CWE-863","CWE-434"}, 0.85),
}
BUDGET = 30         # arbitrary units for one CI run — deliberately tight,
                    # because an unconstrained budget hides the whole problem

def uniform(targets, analyser, budget):
    cost_per = ANALYSERS[analyser][0]
    spend, covered = 0, []
    for t in sorted(targets, key=lambda t: t.name):
        c = cost_per * t.loc / 100
        if spend + c > budget: break
        spend += c; covered.append(t)
    return {"strategy": f"uniform · {analyser}", "spend": round(spend, 1),
            "covered": covered}

def allocated(targets, budget):
    """Deep analysis on high-threat targets, cheap rules everywhere else."""
    ranked = sorted(targets, key=lambda t: -(t.threat_score + t.historical_risk * 3))
    spend, plan = 0.0, []
    for t in ranked:
        for analyser in ("model review", "taint rules", "grep rules"):
            c = ANALYSERS[analyser][0] * t.loc / 100
            wants_deep = (t.threat_score + t.historical_risk * 3) >= 9
            if analyser == "model review" and not wants_deep: continue
            if spend + c <= budget:
                spend += c; plan.append((t, analyser)); break
    return {"strategy": "allocated by threat rank", "spend": round(spend, 1),
            "plan": plan}

u = uniform(TARGETS, "model review", BUDGET)
a = allocated(TARGETS, BUDGET)
print(f"{u['strategy']:34s}spend {u['spend']:>6}  covered {len(u['covered'])}/{len(TARGETS)}")
for t in u["covered"]: print(f"      {t.name}")
print(f"\n{a['strategy']:34s}spend {a['spend']:>6}  covered {len(a['plan'])}/{len(TARGETS)}")
for t, an in a["plan"]: print(f"      {t.name:24s}{an}")

## 4 · Where it breaks — coverage is the wrong metric

In [ ]:
def coverage(plan_targets, targets):
    return len(plan_targets) / len(targets)

def threat_weighted_coverage(plan_targets, targets):
    total = sum(t.threat_score for t in targets)
    got = sum(t.threat_score for t in plan_targets)
    return got / total

u_targets = u["covered"]
a_targets = [t for t, _ in a["plan"]]

print(f"{'strategy':34s}{'coverage':>10}{'threat-weighted':>18}")
print("-" * 64)
for label, ts in (("uniform · model review", u_targets),
                  ("allocated by threat rank", a_targets)):
    print(f"{label:34s}{coverage(ts, TARGETS):>10.0%}{threat_weighted_coverage(ts, TARGETS):>18.0%}")

print("\nThe uniform sweep spent its whole budget alphabetically and covered")
print("the health endpoint before it reached the SQL sink.")
missed = [t.name for t in TARGETS if t not in u_targets and t.threat_score >= 9]
print(f"high-threat targets the uniform sweep never reached: {missed}")
assert missed

## 5 · The control — allocate, then prove the allocation was right

In [ ]:
def plan_report(plan, targets, budget):
    spend = sum(ANALYSERS[an][0] * t.loc / 100 for t, an in plan)
    covered = [t for t, _ in plan]
    deep = [t.name for t, an in plan if an == "model review"]
    uncovered_high = [t.name for t in targets
                      if t not in covered and t.threat_score >= 9]
    return {"budget": budget, "spend": round(spend, 1),
            "threat_weighted_coverage": round(threat_weighted_coverage(covered, targets), 3),
            "deep_analysis_on": deep,
            "uncovered_high_threat": uncovered_high,
            "acceptable": not uncovered_high}

r = plan_report(a["plan"], TARGETS, BUDGET)
for k, v in r.items(): print(f"{k:26s}{v}")
assert r["acceptable"]

print("\nsame budget, if someone doubles the repo with low-risk code:")
BLOAT = TARGETS + [Target(f"vendor_{i}", f"vendor/{i}.py", 1, 0.0, 200)
                   for i in range(1, 9)]
a2 = allocated(BLOAT, BUDGET)
r2 = plan_report(a2["plan"], BLOAT, BUDGET)
print(f"   threat-weighted coverage {r['threat_weighted_coverage']:.0%} → "
      f"{r2['threat_weighted_coverage']:.0%}")
print(f"   uncovered high-threat targets: {r2['uncovered_high_threat'] or 'none'}")
print("\nAllocation is what stops repository growth from silently degrading")
print("the analysis of the parts that matter.")

## 6 · The skill that carries Phase 2

Stages 5 and 6 are now a procedure rather than a one-off. The skill below is the version an agent runs, and its contract is the reason the plan can be handed to Phase 3 without a conversation.

Note what the contract insists on: `score_inputs` alongside every score. A severity you cannot decompose is a severity nobody can argue with — and an unarguable severity is one nobody fixes.

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/appsec/appsec-threat-model/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: appsec-threat-model
description: >-
  Turn an architecture map into a ranked, testable threat model and an audit
  plan. Use after repository reconnaissance, when asked what could go wrong in
  a system, which CWEs apply, where to spend review effort, or how to allocate
  a fixed analysis budget across a codebase.
allowed-tools: Read, Grep, Glob
---

# AppSec pipeline · Phase 2 — Threat modelling and strategy

Covers **stages 5–6**. Consumes `architecture_map` from **appsec-repo-recon**
and produces a ranked threat list plus the plan that spends the audit budget.

A threat model that lists everything is a list, not a model. The output of this
phase is an *ordering*: what to look at first, and what to knowingly skip.

## When to use this

After Phase 1, before any auditing. If asked to "threat model" a system with no
architecture map, run **appsec-repo-recon** first — a threat model built from
file names is fiction.

## Inputs

| Input | Required | Notes |
|---|---|---|
| `architecture_map` | yes | from appsec-repo-recon |
| Audit budget | yes | units of analysis you can actually afford |
| Deployment context | optional | internet-facing vs internal changes exposure |

## Procedure

**Stage 5 — Threat modelling.** For every `reachable` entry→sink pair, ask what
an attacker controlling the entry can do to the sink. Record a threat with:

- the **CWE** it would be, concretely (CWE-89, CWE-22, CWE-78, CWE-502, …)
- the **entry** and **sink** identities, and the path between them
- whether the path crosses a **trust boundary**
- the **authentication** required to reach the entry
- a **score**, computed — never asserted

Score from properties you already have, so the number is reproducible:

```
score = exposure_weight(entry)      # public 3, authenticated 2, internal 1
      × resource_weight(sink)       # subprocess/deserialisation 3, db/fs 2, network 1
      × boundary_multiplier         # 2 if the path crosses a trust boundary
```

A model that outputs a severity without showing the inputs cannot be argued
with, and an unarguable severity is one nobody fixes.

**Stage 6 — Strategic planning.** Spend the budget. Rank threats by score, then
by a **full tiebreak** — `(-score, cwe, entry, sink)` — so equal scores order
identically on every machine. Then allocate:

- Cover the highest-scoring threats first.
- Prefer **breadth across distinct CWEs** over depth on one: three different
  weakness classes found beats three instances of the same one.
- Record what the budget did **not** cover, in `deferred`. An audit plan that
  hides its own gaps produces a report that overclaims.

## Output contract

```json
{
  "threat_model": [
    {"cwe": "CWE-89", "entry": "str", "sink": "str", "path": ["unit"],
     "crosses_boundary": true, "auth": "none|user|admin",
     "score": 0, "score_inputs": {"exposure": 0, "resource": 0, "boundary": 0}}
  ],
  "plan": {
    "budget": 0,
    "selected": [{"threat_index": 0, "cost": 0, "why": "str"}],
    "deferred": [{"threat_index": 0, "why": "str"}],
    "coverage": {"threat_weighted": 0.0, "distinct_cwes": 0}
  }
}
```

`score_inputs` is not optional. It is what makes the ranking reviewable.

## Failure modes

- **Scoring by vibes.** If you cannot show the multiplication, do not emit the
  score.
- **A budget that covers everything.** Then the plan proves nothing — the
  interesting behaviour of an allocator only appears under scarcity. If the
  budget genuinely covers all targets, say so rather than claiming a strategy
  worked.
- **Ranking instability.** Equal scores must not reorder between runs. Sort
  keys first; never iterate a set into a stable sort.
- **Treating "no auth" as the only risk.** An authenticated path to a
  subprocess sink usually outranks an anonymous path to a read-only one.

## Handoff

Pass `threat_model` and `plan.selected` to **appsec-vuln-audit**. Carry
`deferred` all the way to the report — it is the honest scope statement.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
contract = contract_of(body)

# The weakness class each target would be, named rather than implied — the
# contract needs it and "distinct CWEs covered" is meaningless without it.
CWE_OF = {"load_report → execute": "CWE-89", "store → open": "CWE-22",
          "render": "CWE-79", "health": "CWE-200",
          "upload_doc": "CWE-434", "get_report": "CWE-22"}

def cost_of(target, analyser):
    return ANALYSERS[analyser][0] * target.loc / 100

# The plan this lesson produced, in the shape the skill promises.
# threat_index points into `threat_model` below, which is TARGETS order.
# a["plan"] is in *ranked* order, so enumerating it would number the threats
# by rank and every index in the contract would point at the wrong threat.
IDX = {t.name: i for i, t in enumerate(TARGETS)}
sel = [{"threat_index": IDX[t.name], "cost": cost_of(t, analyser),
        "why": analyser}
       for t, analyser in a["plan"]]
chosen = {t.name for t, _ in a["plan"]}
plan = {
 "threat_model": [
   {"cwe": CWE_OF[t.name], "entry": t.name.split(" ")[0], "sink": t.name.split(" ")[-1],
    "path": t.name.split(" → "), "crosses_boundary": t.threat_score >= 6,
    "auth": "none" if t.threat_score >= 8 else "user",
    "score": t.threat_score,
    "score_inputs": {"exposure": t.threat_score,
                     "resource": round(t.historical_risk, 2),
                     "boundary": 2 if t.threat_score >= 6 else 1}}
   for t in TARGETS],
 # What the budget deferred is *depth*, not targets: everything gets some
 # analyser, but the ones that wanted model review and got taint rules are
 # exactly the gap the report must disclose.
 "plan": {"budget": float(BUDGET), "selected": sel,
          "deferred": [{"threat_index": IDX[t.name],
                        "why": f"wanted model review, budget allowed {analyser}"}
                       for t, analyser in a["plan"]
                       if (t.threat_score + t.historical_risk * 3) >= 9
                       and analyser != "model review"],
          "coverage": {"threat_weighted": float(r2["threat_weighted_coverage"]),
                       "distinct_cwes": len({CWE_OF[t.name] for t, _ in a["plan"]})}},
}
problems = check(plan, contract)
print(f"conformance: {len(problems)} problem(s)")
for p in problems: print("   ", p)
assert not problems, problems

print(f"\nselected {len(sel)} of {len(TARGETS)} threats; "
      f"{len(plan['plan']['deferred'])} of them wanted deep review and did not get it")
print(f"distinct CWEs covered: {plan['plan']['coverage']['distinct_cwes']}")
print()
for d in plan["plan"]["deferred"]:
    print(f"   deferred: {TARGETS[d['threat_index']].name:22s} {d['why']}")
print()
print("`deferred` is not bookkeeping. Every target got *an* analyser, so a")
print("coverage number counting targets would read 100%. What the budget")
print("actually cut was depth, on three of the four highest-threat targets.")
print("That distinction travels to the report as the scope statement, and a")
print("plan that drops it produces a report that overclaims.")
assert plan["plan"]["deferred"], "a budget that defers nothing proves nothing"

## What you just proved

On a tight budget the uniform model-review sweep covers only 2 of 6 targets — alphabetically, so it reaches the health endpoint before the SQL sink — giving 33% coverage but only 27% threat-weighted coverage, and missing all three high-threat targets. The allocated plan covers all six within the same budget at 100% threat-weighted coverage, puts deep model review on the SQL sink, and still holds 90% when the repository doubles in size with low-risk code.

## Your turn

Compute threat-weighted coverage for your current scanning setup. If you scan everything uniformly, the number equals your raw coverage — which means you have no allocation strategy, only a budget that will eventually be cut.

---

**Next → [B1.5 · Vulnerability auditing: three generations of SAST](https://spbreed.github.io/cyber-commons/lessons/B1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*